In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from torchvision import transforms
from PIL import Image
from tqdm.notebook import tqdm
from sklearn.metrics import accuracy_score, f1_score



In [2]:
from model import CNN
from data import GameplayDatasetCNN

In [3]:
EPOCHS = 15

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = CNN(in_channels=1,num_classes=3,channels=16,final_pool=8)
model = model.to(device)


In [5]:
train_dataset = GameplayDatasetCNN()
val_dataset = GameplayDatasetCNN(False)

train_dataloader = DataLoader(dataset=train_dataset,
                              batch_size=64,
                              shuffle=True,
                              num_workers=0)

val_dataloader = DataLoader(dataset=val_dataset,
                            batch_size=64,
                            shuffle=True,
                            num_workers=0)

In [6]:
optimizer = torch.optim.Adam(model.parameters(),lr=1e-3)

def train(epoch):
    model.train()
    train_loss = 0
    for batch_idx, (input,output) in tqdm(enumerate(train_dataloader),total=len(train_dataloader)):
        optimizer.zero_grad()
        input = input.to(device)
        output = output.to(device)
        y_pred = model(input)
        loss = F.cross_entropy(y_pred,output)
        loss.backward()
        train_loss += loss.item()
        optimizer.step()
        

    print(f'Epoch {epoch}: Average loss: {train_loss / (len(train_dataloader)):.4f}') 
    print("Evaluation")
    model.eval()
    predictions = []
    truths = []

    for x,y in tqdm(val_dataloader,total=len(val_dataloader)):
        x = x.to(device)
        y = y.to(device)
        pred = torch.argmax(model(x),dim=-1).tolist()
        predictions.extend(pred)
        truths.extend(y.tolist())

    print(f"Validation Accuracy : {accuracy_score(truths,predictions)} | Validation F1 Score : {f1_score(truths,predictions,average='macro')}")   
    return f1_score(truths,predictions,average='macro')

In [7]:
best_val = 0
scores = []


for i in range(1,EPOCHS+1):
    val_metric = train(i)
    scores.append(val_metric)
    if val_metric > best_val:
        print(f"New checkpoint saved!")
        torch.save(model.state_dict(),'model.pth')
        best_val = val_metric

    if len(scores) > 3:
        if val_metric < scores[-4] and scores[-2] < scores[-4] and scores[-3] < scores[-4]:
            break
    

  0%|          | 0/189 [00:00<?, ?it/s]

Epoch 1: Average loss: 0.3020
Evaluation


  0%|          | 0/48 [00:00<?, ?it/s]

Validation Accuracy : 0.8875248838752489 | Validation F1 Score : 0.8802500016393039
New checkpoint saved!


  0%|          | 0/189 [00:00<?, ?it/s]

...

Validation Accuracy : 0.8002654280026543 | Validation F1 Score : 0.7977530050180265


  0%|          | 0/189 [00:00<?, ?it/s]

Epoch 12: Average loss: 0.0888
Evaluation


  0%|          | 0/48 [00:00<?, ?it/s]

Validation Accuracy : 0.948905109489051 | Validation F1 Score : 0.9471758826989892
